## Data Preparation, Cleaning, and Preprocessing

In [ ]:
import pandas as pd
import os
from typing import Dict, Tuple

class DataPreparator:
    """Data Preparator - Reuses original logic for data loading, merging, and splitting"""

    def __init__(self, config):
        self.config = config
        self.base_path = config['data']['base_path']
        self.raw_comment_file = "資料集總表 - Comment 總表.csv"
        self.raw_post_file = "資料集總表 - Post 總表.csv"

    def load_raw_data(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Load raw comment and post data"""
        print("載入原始資料...")
        print("=" * 40)

        # Load comment data
        comment_path = os.path.join(self.base_path, 'data', self.raw_comment_file)
        if not os.path.exists(comment_path):
            raise FileNotFoundError(f"找不到評論檔案: {comment_path}")

        comment_table = pd.read_csv(comment_path, encoding='utf-8-sig')
        print(f"載入評論資料: {len(comment_table)} 筆")

        # Load post data
        post_path = os.path.join(self.base_path, 'data', self.raw_post_file)
        if not os.path.exists(post_path):
            raise FileNotFoundError(f"找不到貼文檔案: {post_path}")

        post_table = pd.read_csv(post_path, encoding='utf-8-sig')
        print(f"載入貼文資料: {len(post_table)} 筆")

        return comment_table, post_table

    def clean_and_merge_data(self, comment_table: pd.DataFrame, post_table: pd.DataFrame) -> pd.DataFrame:
        """Clean and merge comment and post data"""
        print("\n資料清理與合併...")
        print("-" * 30)

        # 1. Remove 'mixed' sentiment labels
        mix_count = len(comment_table[comment_table['sentiment'] == 'mixed'])
        comment_table = comment_table[comment_table['sentiment'] != 'mixed']
        print(f"已移除 {mix_count} 筆 'mixed' 情緒標籤資料")

        # 2. Remove data that cannot be matched to posts
        unmatched_count = len(comment_table[comment_table['sentiment'] == '無法配對至貼文'])
        comment_table = comment_table[comment_table['sentiment'] != '無法配對至貼文']
        print(f"已移除 {unmatched_count} 筆 '無法配對至貼文' 資料")

        # 3. Merge comment and post data
        print(f"\n合併前評論數量: {len(comment_table)}")

        # Select required post columns
        post_columns = ['post_id', '活動貼標', 'YT 帳號分類', '貼文主題', 'created_time (TW)']

        # Check if these columns exist in the post table
        missing_columns = [col for col in post_columns if col not in post_table.columns]
        if missing_columns:
            print(f"警告: 貼文表中缺少欄位: {missing_columns}")
            post_columns = [col for col in post_columns if col in post_table.columns]

        # Execute merge
        full_data = pd.merge(comment_table, post_table[post_columns], on='post_id', how='inner')

        unmatched_post_count = len(comment_table) - len(full_data)
        print(f"合併後評論數量: {len(full_data)}")
        print(f"無法配對至貼文: {unmatched_post_count} 筆")

        return full_data

    def split_datasets(self, full_data: pd.DataFrame) -> Dict[str, pd.DataFrame]:
        """Split dataset into 4 target datasets"""
        print("\n切分資料集...")
        print("-" * 30)

        datasets = {}

        # 1. Operations/Events related data
        print("處理營運活動資料...")

        # Get operations related posts (based on post theme)
        operation_posts = self.post_table[self.post_table['貼文主題'] == '營運活動']['post_id'].unique()
        operation_data = full_data[full_data['post_id'].isin(operation_posts)]

        print(f"營運活動總留言數: {len(operation_data)}")

        # Separate positive and negative sentiments
        operation_positive = operation_data[operation_data['sentiment'] == 'positive']
        operation_negative = operation_data[operation_data['sentiment'] == 'negative']

        datasets['operation_positive'] = operation_positive
        datasets['operation_negative'] = operation_negative

        print(f"  - 營運活動正面情緒: {len(operation_positive)} 筆")
        print(f"  - 營運活動負面情緒: {len(operation_negative)} 筆")

        # 2. IP Collaboration related data
        print("\n處理IP聯動資料...")

        # Condition 1: Post theme = "IP Collaboration"
        ip_theme_posts = self.post_table[self.post_table['貼文主題'] == 'IP 聯動']['post_id'].unique()

        # Condition 2: Activity tags = "Bleach Collab", "Hunter Collab", "Jujutsu Collab"
        ip_activity_posts = self.post_table[self.post_table['活動貼標'].isin(['死神聯動', '獵人聯動', '咒術聯動'])]['post_id'].unique()

        # Merge post_ids from both conditions
        all_ip_posts = set(ip_theme_posts) | set(ip_activity_posts)
        ip_data = full_data[full_data['post_id'].isin(all_ip_posts)]

        print(f"IP聯動總留言數: {len(ip_data)}")

        # Separate positive and negative sentiments
        ip_positive = ip_data[ip_data['sentiment'] == 'positive']
        ip_negative = ip_data[ip_data['sentiment'] == 'negative']

        datasets['ip_positive'] = ip_positive
        datasets['ip_negative'] = ip_negative

        print(f"  - IP聯動正面情緒: {len(ip_positive)} 筆")
        print(f"  - IP聯動負面情緒: {len(ip_negative)} 筆")

        # 3. Balance Update related data
        print("\n處理平衡性版本更新資料...")

        # Filter based on post theme
        balance_posts = self.post_table[self.post_table['貼文主題'] == '平衡性版本更新']['post_id'].unique()
        balance_data = full_data[full_data['post_id'].isin(balance_posts)]

        print(f"平衡性版本更新總留言數: {len(balance_data)}")

        datasets['balance_update'] = balance_data
        print(f"  - 平衡性版本更新: {len(balance_data)} 筆")

        # 4. Data validation
        print(f"\n資料切分驗證:")
        total_split = sum(len(df) for df in datasets.values())
        print(f"原始合併資料: {len(full_data)} 筆")
        print(f"切分後總計: {total_split} 筆")

        return datasets

    def prepare_final_datasets(self, datasets: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
        """Prepare final dataset format"""
        print("\n準備最終資料集...")
        print("-" * 30)

        final_datasets = {}

        # Define output columns and renaming rules
        standard_columns = ['comment_id', 'post_id', 'cleaned_comment', 'sentiment', '活動貼標']
        column_rename = {
            'cleaned_comment': 'text',
            'sentiment': 'label',
            '活動貼標': 'activity'
        }

        # Use more columns for specific datasets
        extended_columns = ['comment_id', 'post_id', 'cleaned_comment', 'sentiment', '活動貼標', 'created_time (TW)']
        extended_rename = {
            'cleaned_comment': 'text',
            'sentiment': 'label',
            '活動貼標': 'activity',
            'created_time (TW)': 'created_time_tw'
        }

        for dataset_name, df in datasets.items():
            print(f"處理 {dataset_name}...")

            # Select columns (some datasets need time columns)
            if dataset_name in ['operation_positive', 'operation_negative', 'ip_positive', 'ip_negative', 'balance_update']:
                # Check if time column exists
                if 'created_time (TW)' in df.columns:
                    use_columns = extended_columns
                    use_rename = extended_rename
                else:
                    use_columns = standard_columns
                    use_rename = column_rename
            else:
                use_columns = standard_columns
                use_rename = column_rename

            # Check if necessary columns exist
            missing_cols = [col for col in use_columns if col not in df.columns]
            if missing_cols:
                print(f"  警告: 缺少欄位 {missing_cols}，使用可用欄位")
                use_columns = [col for col in use_columns if col in df.columns]

            # Select columns and rename
            if use_columns:
                final_df = df[use_columns].copy()
                final_df = final_df.rename(columns=use_rename)
                final_datasets[dataset_name] = final_df
                print(f"  - 最終欄位: {list(final_df.columns)}")
                print(f"  - 資料筆數: {len(final_df)}")
            else:
                print(f"  錯誤: 無可用欄位")

        return final_datasets

    def save_datasets(self, datasets: Dict[str, pd.DataFrame]) -> Dict[str, str]:
        """Save all datasets"""
        print("\n儲存資料集...")
        print("-" * 30)

        data_dir = os.path.join(self.base_path, 'data')
        os.makedirs(data_dir, exist_ok=True)

        saved_paths = {}

        # Filename mapping
        filename_mapping = {
            'operation_positive': 'operation_positive_data.csv',
            'operation_negative': 'operation_negative_data.csv',
            'ip_positive': 'positive_ip_collaboration_data.csv',
            'ip_negative': 'negative_ip_collaboration_data.csv',
            'balance_update': 'balance_update_data.csv'
        }

        for dataset_name, df in datasets.items():
            filename = filename_mapping.get(dataset_name, f"{dataset_name}_data.csv")
            file_path = os.path.join(data_dir, filename)

            df.to_csv(file_path, index=False, encoding='utf-8-sig')
            saved_paths[dataset_name] = file_path

            print(f"已儲存 {dataset_name}: {file_path} ({len(df)} 筆)")

        return saved_paths

    def run_full_preparation(self) -> Dict[str, str]:
        """Execute the complete data preparation process"""
        print("開始資料準備流程")
        print("=" * 50)

        try:
            # 1. Load raw data
            comment_table, post_table = self.load_raw_data()

            # Set post_table as instance variable for use in other methods
            self.post_table = post_table

            # 2. Clean and merge data
            full_data = self.clean_and_merge_data(comment_table, post_table)

            # 3. Split datasets
            datasets = self.split_datasets(full_data)

            # 4. Prepare final format
            final_datasets = self.prepare_final_datasets(datasets)

            # 5. Save datasets
            saved_paths = self.save_datasets(final_datasets)

            # 6. Validate results
            self.validate_results(final_datasets)

            print(f"\n資料準備完成！")
            print(f"共生成 {len(saved_paths)} 個資料集檔案")

            return saved_paths

        except Exception as e:
            print(f"資料準備過程出錯: {str(e)}")
            raise

    def validate_results(self, datasets: Dict[str, pd.DataFrame]):
        """Validate final results"""
        print("\n資料驗證...")
        print("-" * 30)

        for dataset_name, df in datasets.items():
            print(f"\n{dataset_name}:")
            print(f"  - 資料筆數: {len(df)}")
            print(f"  - 欄位: {list(df.columns)}")

            # Check text length
            if 'text' in df.columns:
                avg_length = df['text'].str.len().mean()
                print(f"  - 平均文本長度: {avg_length:.1f} 字")

                # Check for empty texts
                empty_texts = df['text'].isna().sum() + (df['text'] == '').sum()
                if empty_texts > 0:
                    print(f"  - 警告: 發現 {empty_texts} 個空文本")

            # Check sentiment distribution (if label column exists)
            if 'label' in df.columns:
                print(f"  - 情緒分布: {df['label'].value_counts().to_dict()}")

# Initialize DataPreparator
preparator = DataPreparator(PIPELINE_CONFIG)
print("DataPreparator initialized successfully")

DataPreparator initialized successfully


In [ ]:
def prepare_all_data():
    """Execute complete data preparation process"""
    try:
        saved_paths = preparator.run_full_preparation()

        print("\n" + "="*50)
        print("資料準備成功完成！")
        print("="*50)
        print("生成的檔案:")
        for dataset_name, path in saved_paths.items():
            print(f"  - {dataset_name}: {os.path.basename(path)}")

        print(f"\n現在可以執行後續的模型訓練步驟")
        print("生成的資料集:")
        print("- operation_positive: 營運活動正面情緒")
        print("- operation_negative: 營運活動負面情緒")
        print("- ip_positive: IP聯動正面情緒")
        print("- ip_negative: IP聯動負面情緒")
        print("- balance_update: 平衡性版本更新")
        return saved_paths

    except Exception as e:
        print(f"資料準備失敗: {str(e)}")
        return None

# Execute data preparation (uncomment to run)
saved_paths = prepare_all_data()

開始資料準備流程
載入原始資料...
載入評論資料: 165704 筆
載入貼文資料: 6062 筆

資料清理與合併...
------------------------------
已移除 1 筆 'mixed' 情緒標籤資料
已移除 1469 筆 '無法配對至貼文' 資料

合併前評論數量: 164234
合併後評論數量: 164234
無法配對至貼文: 0 筆

切分資料集...
------------------------------
處理營運活動資料...
營運活動總留言數: 4133
  - 營運活動正面情緒: 918 筆
  - 營運活動負面情緒: 1044 筆

處理IP聯動資料...
IP聯動總留言數: 12612
  - IP聯動正面情緒: 5035 筆
  - IP聯動負面情緒: 2571 筆

處理平衡性版本更新資料...
平衡性版本更新總留言數: 22906
  - 平衡性版本更新: 22906 筆

資料切分驗證:
原始合併資料: 164234 筆
切分後總計: 32474 筆

準備最終資料集...
------------------------------
處理 operation_positive...
  - 最終欄位: ['comment_id', 'post_id', 'text', 'label', 'activity', 'created_time_tw']
  - 資料筆數: 918
處理 operation_negative...
  - 最終欄位: ['comment_id', 'post_id', 'text', 'label', 'activity', 'created_time_tw']
  - 資料筆數: 1044
處理 ip_positive...
  - 最終欄位: ['comment_id', 'post_id', 'text', 'label', 'activity', 'created_time_tw']
  - 資料筆數: 5035
處理 ip_negative...
  - 最終欄位: ['comment_id', 'post_id', 'text', 'label', 'activity', 'created_time_tw']
  - 資料筆數: 2571
處理 balance_u

In [ ]:
import warnings
from ckip_transformers.nlp import CkipWordSegmenter
import stopwordsiso as siso

class SocialMediaAnalyzer:
    """Social Media Text Analyzer"""

    def __init__(self, language="multilingual", output_dir="results", use_gpu=True):
        self.language = language
        self.output_dir = output_dir
        self.use_gpu = use_gpu
        os.makedirs(output_dir, exist_ok=True)
        self._init_tools()

    def _init_tools(self):
        """Initialize required tools"""
        # Load embedding model
        try:
            from sentence_transformers import SentenceTransformer
            if self.language == "chinese" or self.language == "multilingual":
                self.embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
            else:
                self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
            print(f"成功載入嵌入模型")
        except ImportError:
            warnings.warn("無法載入sentence_transformers，請安裝相關套件")
            self.embedding_model = None

        # Load stopwords
        default_zh_stopwords = self._get_default_chinese_stopwords()
        default_en_stopwords = self._get_default_english_stopwords()

        try:
            self.stopwords = {
                "zh": set(siso.stopwords("zh")) | default_zh_stopwords,
                "en": set(siso.stopwords("en")) | default_en_stopwords
            }
            self.stopwordsiso_loaded = True
            print(f"成功載入stopwordsiso並合併自定義停用詞，中文停用詞總數: {len(self.stopwords['zh'])}")
        except ImportError:
            warnings.warn("無法載入stopwordsiso，使用自定義停用詞")
            self.stopwordsiso_loaded = False
            self.stopwords = {
                "zh": default_zh_stopwords,
                "en": default_en_stopwords
            }
            print(f"使用自定義停用詞，中文停用詞總數: {len(self.stopwords['zh'])}")

        # Initialize preserved terms set
        self.preserved_terms = set()

        # Load CKIP segmentation tool
        try:
            device = 0 if self.use_gpu else -1
            self.ws_driver = CkipWordSegmenter(model="bert-base", device=device)
            self.ckip_loaded = True
            print(f"成功載入CKIP分詞模型 (設備: {'GPU' if device==0 else 'CPU'})")
        except ImportError:
            warnings.warn("無法載入CKIP分詞工具，中文分詞功能將受限")
            self.ckip_loaded = False

    def _get_default_chinese_stopwords(self) -> set:
        """Get default Chinese stopwords"""
        return {
            '嗎', '欸', '喔', '啊', '哦', '哈哈', '哈哈哈', '笑死', '好笑', '一下', '的', '了',
            '是', '吧', '於', '之', '也', '都', '就', '而', '這', '那', '還有', '還是', '只能',
            '可以', '不要', '不過', '感謝', '謝謝', '可能', '真的', '因為', '所以', '其實', '如果',
            '但是', '總是', '基本', '突然', '一般', '開始', '沒有', '有點', '其他', '甚麼', '什麼',
            '加油', '呢', '啦', '嘛', '耶', '阿', '拉', '欸', '就是', '這樣', '那個', '然後','還是','好像',
            '可是', '哭哭', '嘻嘻', '呵呵', '嘿嘿', '耶耶', 'qvzl6ngg', 'rtgfradbg', 'jt', '其實', '以為', '還', '這樣',
            '我', '你', '不', '有', '一', '個', '在', '好', '很', '要', '會', '那', '人', '還',
            '說', '看', '跟', '他', '能', '打', '玩', '被', '太', '但', '去', '才', '想', '上',
            '到', '多', '只', '用', '給', '買', '大', '不會', '現在', '怎麼', '最', '過', '自己',
            '覺得', '來', '又', '知道', '種', '再', '比較', '出', '應該', '把', '直接', '讓', '超',
            '如果', '可能', '好像', '得', '中', '不過', '然而', '或許', '大概', '看起來', '感覺上',
            '完全', '根本', '幾乎', '差不多', '到底', '給', '太', '講', '幾', '次',
            '場', '幫', '哈', '看到', '知道', '恭喜', '對',
            '像', '你們', '點', '問題', '誰', '一樣', '一直', '沒', '搞', '會不會', '笑',
            '每次', '拜託', '整個', '之前', '特別', '明明', '這種', '已經', '總是', '沒用',
            '不用', '不是', '不能', '還有', '真是', '變成', '進去','⁎⁍̴̛ᴗ⁍̴̛⁎',
            '逼','連', '給','欸欸', '欸欸欸', '欸', '嗚', '有點',' 因為',' 雖然',
            '不可能', '絕對不會', '還不如', '告訴', '我們', '這麼', '那麼', '為什麼',
            '欸欸欸', '欸欸', '嘟嘟', '嗚哇', '嗯哼', '嗚',
            '分鐘', '前', '時間', '小時', '以前', '之後', '已經',
            '&zhg', '&hg', '&m@', 'jfg', 'gg', 'ngg',
            '而且', '只有', '只是', '的話', '從', '比', '更', '呢', '喔',
            '時候', '全', '每', '很多', '前', '後', '第一',
            '和', '選', '搭配', '以為', '請問', '終於',
        }

    def _get_default_english_stopwords(self) -> set:
        """Get default English stopwords"""
        return {
            'the', 'and', 'is', 'of', 'to', 'in', 'that', 'it', 'for',
            'you', 'was', 'with', 'on', 'as', 'have', 'are', 'this',
            'be', 'at', 'but', 'or', 'from', 'had', 'by', 'not', 'what',
            'all', 'were', 'we', 'when', 'your', 'can', 'said', 'there',
            'use', 'an', 'each', 'which', 'do', 'how', 'their', 'if',
            'will', 'up', 'about', 'out', 'many', 'then', 'them', 'these',
            'so', 'some', 'her', 'would', 'make', 'like', 'him', 'into',
            'time', 'has', 'look', 'two', 'more', 'go', 'see', 'no', 'way',
            'could', 'my', 'than', 'been', 'who', 'its', 'now', 'did'
        }

    def batch_tokenize(self, texts: List[str], min_token_length: int = 1, batch_size: int = 1500) -> List[List[str]]:
        """Batch process tokenization for multiple texts - Optimized for CKIP"""
        if not texts:
            return []

        if not self.ckip_loaded:
            return [self.multilingual_tokenizer(text, min_token_length) for text in texts]

        processed_texts = []
        chinese_texts = []

        for text in texts:
            if bool(re.search('[\u4e00-\u9fff]', text)):
                chinese_texts.append(text)
            else:
                processed_texts.append(self.multilingual_tokenizer(text, min_token_length))

        if not chinese_texts:
            return processed_texts

        ckip_results = []
        stopwords = self.stopwords["zh"]

        for i in range(0, len(chinese_texts), batch_size):
            batch = chinese_texts[i:min(i+batch_size, len(chinese_texts))]

            try:
                word_sentences = self.ws_driver(batch)

                for word_sentence in word_sentences:
                    tokens = []
                    for w in word_sentence:
                        if w in self.preserved_terms:
                            tokens.append(w)
                        elif (len(w.strip()) >= min_token_length and
                              not re.fullmatch(r'[^\w\u4e00-\u9fff]+', w) and
                              w.lower() not in stopwords):
                            tokens.append(w)

                    ckip_results.append(tokens if tokens else [])

            except Exception as e:
                warnings.warn(f"CKIP批次分詞失敗: {str(e)}，使用備用分詞")
                for text in batch:
                    tokens = []
                    processed_text = text
                    for term in self.preserved_terms:
                        if term in processed_text:
                            processed_text = processed_text.replace(term, " <TERM> ")
                            tokens.append(term)

                    other_parts = re.findall(r'\w+', processed_text)
                    for part in other_parts:
                        if len(part) >= min_token_length and part not in stopwords:
                            tokens.append(part)

                    ckip_results.append(tokens if tokens else [])

        # Merge results
        chinese_idx = 0
        non_chinese_idx = 0
        combined_results = []

        for text in texts:
            if bool(re.search('[\u4e00-\u9fff]', text)):
                if chinese_idx < len(ckip_results):
                    combined_results.append(ckip_results[chinese_idx])
                    chinese_idx += 1
                else:
                    combined_results.append([])
            else:
                if non_chinese_idx < len(processed_texts):
                    combined_results.append(processed_texts[non_chinese_idx])
                    non_chinese_idx += 1
                else:
                    combined_results.append([])

        return combined_results

    def multilingual_tokenizer(self, text: str, min_token_length: int = 1) -> List[str]:
        """Multilingual tokenizer for single text"""
        has_chinese = bool(re.search('[\u4e00-\u9fff]', text))
        stopwords = self.stopwords["zh"] if has_chinese else self.stopwords["en"]

        if has_chinese and self.ckip_loaded:
            return self.batch_tokenize([text], min_token_length)[0]

        tokens = []
        english_tokens = re.findall(r'\b\w+\b', text.lower())
        for t in english_tokens:
            if t in self.preserved_terms:
                tokens.append(t)
            elif (len(t) >= min_token_length and
                  t not in stopwords and
                  not re.fullmatch(r'[^\w]+', t)):
                tokens.append(t)

        if has_chinese:
            chinese_chars = re.findall('[\u4e00-\u9fff]+', text)
            for char_group in chinese_chars:
                if char_group in self.preserved_terms:
                    tokens.append(char_group)
                elif len(char_group) == 1:
                    if char_group not in stopwords:
                        tokens.append(char_group)
                else:
                    i = 0
                    while i < len(char_group) - 1:
                        bigram = char_group[i:i+2]
                        if bigram in self.preserved_terms:
                            tokens.append(bigram)
                            i += 2
                        elif bigram not in stopwords:
                            tokens.append(bigram)
                            i += 1
                        else:
                            i += 1

        return tokens if tokens else [text]

class BatchTokenizer:
    """Batch tokenizer for BERTopic"""
    def __init__(self, analyzer, batch_size):
        self.analyzer = analyzer
        self.batch_size = batch_size

    def __call__(self, texts):
        if isinstance(texts, str):
            return self.analyzer.multilingual_tokenizer(texts)
        return self.analyzer.batch_tokenize(texts, batch_size=self.batch_size)

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.base_path = config['data']['base_path']

    def clean_text(self, text, is_forum_post=False):
        """Fine-grained text cleaning - Reuse original logic"""
        if pd.isna(text) or text == "":
            return ""

        # Filter out all English for posts from Bahamut forum
        if is_forum_post:
            text = re.sub(r'[a-zA-Z]', ' ', text)

        # Remove patterns for invitation codes and share links
        text = re.sub(r'[a-zA-Z0-9+/]+https?://coinpusher\.moba\.garena\.tw[^\s]*', ' ', text)

        # Remove patterns with alphanumeric characters around @ symbol
        text = re.sub(r'[a-zA-Z0-9]+@[a-zA-Z0-9]+', ' ', text)

        # Remove all URLs
        text = re.sub(r'https?://\S+', ' ', text)
        text = re.sub(r'www\.\S+', ' ', text)
        text = re.sub(r'm\.gamer\.com\.\S+', ' ', text)

        # Remove time format
        text = re.sub(r'\b([01]?[0-9]|2[0-3]):([0-5][0-9])(:[0-5][0-9])?\b', ' ', text)

        # Remove punctuation
        punctuation = r'，。！？；：""''（）【】『』「」《》〈〉…—～,.!?;:"\'`()[]{}·<>'
        text = re.sub(f'[{re.escape(punctuation)}]', ' ', text)

        # Handle excess whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    def load_dataset(self, dataset_name):
        """Load specified dataset"""
        if dataset_name not in self.config['data']['datasets']:
            raise ValueError(f"Dataset {dataset_name} not found in configuration")

        dataset_config = self.config['data']['datasets'][dataset_name]
        file_path = os.path.join(self.base_path, 'data', dataset_config['file'])

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Data file not found: {file_path}")

        # Load data
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        print(f"Loaded {dataset_config['description']}: {len(df)} records")

        # Mark Bahamut forum posts
        df['is_forum_post'] = df['post_id'].str.contains('forum.gamer.com', na=False)
        forum_post_count = df['is_forum_post'].sum()
        print(f"Found {forum_post_count} forum posts requiring special cleaning")

        # Clean text
        if 'text' in df.columns:
            df['cleaned_text'] = df.apply(lambda row: self.clean_text(row['text'], row['is_forum_post']), axis=1)
        else:
            raise ValueError("Column 'text' not found in dataset")

        # Filter empty texts
        df_clean = df[df['cleaned_text'].str.strip() != ""].copy()
        print(f"After cleaning: {len(df_clean)} valid records")

        return df_clean

    def get_texts(self, dataset_name):
        """Get cleaned text list"""
        df = self.load_dataset(dataset_name)
        return df['cleaned_text'].tolist()

# Initialize analyzer and processor
analyzer = SocialMediaAnalyzer(language="multilingual", output_dir="results", use_gpu=True)
processor = DataProcessor(PIPELINE_CONFIG)
print("Data processing components initialized successfully")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/407M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

成功載入CKIP分詞模型 (設備: GPU)
Data processing components initialized successfully
